# Model comparison

Compares two fitting models using the cross-validation $\chi^2$ matrices saved by `Bias_Fitting.ipynb`
(`<write_root>CV_Chi2.npy`, shape `(phases, phases, 2)` for $(\chi^2_P, \chi^2_B)$; entry `[i, j]` is phase $i$'s
best-fit bias parameters scored on phase $j$'s data with the noise parameters refit on phase $j$).

The two models are matched element by element on the off-diagonal entries only (the diagonal is the training $\chi^2$).
The error on the mean $\Delta\chi^2$ is a delete-one jackknife over phases: for each phase $k$, remove row $k$ **and**
column $k$, recompute the mean, and take $\sigma^2 = \frac{n-1}{n}\sum_k (\theta_k - \bar\theta)^2$.

In [1]:
import numpy as np

In [11]:
# Only change the variables in this block. Delta chi2 = model_a - model_b (negative: model_a is preferred).
root = 'New_Cov_w_Cross_no_Bins/v2_'
model_a = 'Reduced'
model_b = 'EFT'
cv_file = '_CV_Chi2.npy'

In [12]:
def jackknife_mean(D):
    """Mean of the off-diagonal entries of D (phases x phases) and its delete-one jackknife error,
    removing row k and column k together for each phase k."""
    n = len(D)
    mean = D[~np.eye(n, dtype=bool)].mean()
    theta = np.zeros(n)
    for k in range(n):
        keep = np.delete(np.arange(n), k)
        theta[k] = D[np.ix_(keep, keep)][~np.eye(n - 1, dtype=bool)].mean()
    return mean, np.sqrt((n - 1) / n * np.sum((theta - theta.mean())**2))

In [13]:
cv_a = np.load(root + model_a + cv_file)
cv_b = np.load(root + model_b + cv_file)
phases = len(cv_a)
off = ~np.eye(phases, dtype=bool)

print(f'Delta chi2 = {model_a} - {model_b}  (negative: {model_a} preferred)')
for label, D in [('P', cv_a[..., 0] - cv_b[..., 0]),
                 ('B', cv_a[..., 1] - cv_b[..., 1]),
                 ('P + B', cv_a.sum(2) - cv_b.sum(2))]:
    mean, err = jackknife_mean(D)
    # Number of fit phases i whose mean test chi2 (over j != i) is lower for model_a.
    wins = (np.nanmean(np.where(off, D, np.nan), axis=1) < 0).sum()
    print(f'{label:6s} {mean:8.2f} +- {err:5.2f}   ({mean/err:5.1f} sigma, {model_a} better for {wins}/{phases} fit phases)')

Delta chi2 = Reduced - EFT  (negative: Reduced preferred)
P          2.39 +-  1.11   (  2.1 sigma, Reduced better for 6/25 fit phases)
B         -2.78 +-  1.25   ( -2.2 sigma, Reduced better for 16/25 fit phases)
P + B      0.64 +-  1.27   (  0.5 sigma, Reduced better for 8/25 fit phases)
